# 04 · Modeling: nested, grouped cross-validation

In this notebook I fit and evaluate six kinds of model on the same four problems I set myself in 2022: predicting the kingdom (strictly, the CUTG division: 10 classes) and the DNA type (3 classes), each from either the 64 codon frequencies or the 21 amino-acid groups. What has changed is how the models are tested, which matters more here than the choice of model.

**How the models are tested**

- *Repeated cross-validation.* Each model is scored on 15 test folds: 5-fold cross-validation, repeated 3 times with different shuffles. Fifteen scores give a mean and a spread, which a single split cannot. Every model and feature set sees exactly the same folds, so they can be compared pair by pair, fold by fold.
- *Three ways of building the folds.* For the main results I hold out whole species (`SpeciesID`), so a model is always tested on organisms absent from its training data. Holding out whole genus groups is a stricter test, which removes close relatives too. Ordinary random folds, which ignore relatedness, I use only to measure how much the grouping matters. Folds are stratified, so each keeps roughly the class proportions of the whole.
- *Nested tuning.* Where a model has a setting worth tuning (k for k-NN, the smoothing for naive Bayes), it is chosen by a second, inner cross-validation run inside each outer training set, scored by macro-F1. The outer test fold is never used to choose anything, so its score remains an unbiased estimate.
- *The models.* A majority-class baseline (always predict the most common class), Gaussian naive Bayes, k-NN (distance-weighted, on scaled features), a random forest (200 trees), XGBoost (150 rounds) and an MLP (one hidden layer of 128 units, on scaled features, with early stopping). The last three use fixed, conventional settings (`codon_utils.make_models`); with nested cross-validation, every extra tuned setting multiplies the running time, and these defaults are reasonable for data of this size.
- *The metrics.* For each test fold I record accuracy, balanced accuracy, macro-F1 and ROC-AUC (one-vs-rest, from the predicted probabilities). Macro-F1 is the headline. It computes the F1 score of each class separately and averages them, so archaea (126 rows) count as much as bacteria (2,918); accuracy, by contrast, is dominated by the large classes, and a model could neglect the small ones almost entirely without much penalty.

**What I changed from 2022, and why.** Originally I set aside a single test split and then chose k, and the number of trees, by their scores on that same test set. That makes the reported scores optimistic (the test set was used to select the model it then evaluated), and one split says nothing about variability. Nested, repeated cross-validation fixes both problems.

I've also dropped Hyperopt. My objective function converted `colsample_bytree` to an integer, which turned every value drawn between 0.5 and 1 into 0, and XGBoost accepts 0 silently and grows each tree on a single feature; the function also sampled `reg_lambda` and then never passed it to the model. For the few settings that matter here, a small explicit grid is simpler, and every candidate is visible.

The SHAP plots have been dropped: they were computed on the training rows of a single split and did not bear on these questions.

**Runtime.** The full run takes about 40 minutes on 2 CPU cores. Setting the environment variable `CODON_SMOKE=1` runs a tiny version instead (1 repeat × 3 folds, three models, species folds only), which takes about a minute and writes to `results/smoke/`, which git ignores. It exists to check that everything runs end to end, and its numbers shouldn't be read as results.

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd

import codon_utils

SMOKE = os.environ.get('CODON_SMOKE') == '1'
OUT = Path('../results/smoke' if SMOKE else '../results')
OUT.mkdir(parents=True, exist_ok=True)

if SMOKE:  # quick end-to-end check (CI): about a minute
    SCHEMES, REPEATS, FOLDS, MODELS = ['species'], 1, 3, ['majority', 'knn', 'xgb']
else:      # the full design: about 40 minutes on 2 CPU cores
    SCHEMES, REPEATS, FOLDS = ['species', 'genus', 'ungrouped'], 3, 5
    MODELS = ['majority', 'gnb', 'knn', 'rf', 'xgb', 'mlp']
print('smoke run' if SMOKE else 'full run', '-> writing to', OUT)

## Data

The rows, features and groups of `03_preprocessing`, rebuilt by `codon_utils.load_project` from the raw file, so this notebook depends on no saved intermediate file. The row counts table records how the 13,028 raw rows became 12,963.

In [ ]:
df, row_counts = codon_utils.load_project('../data/raw/codon_usage.csv')
row_counts.to_csv(OUT / 'row_counts.csv', index=False)

FEATURES = {'codon': df[codon_utils.CODONS], 'amino': codon_utils.amino_features(df)}
TASKS = {'kingdom': df['kingdom'], 'dna_type': df['dna_type']}
GROUPS = {'species': df['group_species'], 'genus': df['group_genus'],
          'ungrouped': df['group_species']}  # the ungrouped splitter ignores groups
row_counts

## Models

Each model, with the grid its inner cross-validation searches; "fixed settings" means that nothing is searched.

In [ ]:
models = {name: spec for name, spec in codon_utils.make_models(seed=0).items() if name in MODELS}
for name, (estimator, grid) in models.items():
    print(f'{name:9s}', grid or 'fixed settings')

## Run

The main loop: one call to `codon_utils.nested_cv` for each fold scheme, task and feature set, 12 calls in all. Repeat *r* uses outer seed *r* and inner seed 100 + *r*, so every model and feature set sees identical folds, and the whole run is reproducible. For the species scheme I also keep the out-of-fold predictions (each row's prediction from the model that didn't see it), which `05_results` turns into a confusion matrix.

In [ ]:
scores, params, oof = [], [], []
start = time.perf_counter()
for scheme in SCHEMES:
    for task, y in TASKS.items():
        for features, X in FEATURES.items():
            s, p, o = codon_utils.nested_cv(X, y, GROUPS[scheme], models, scheme=scheme,
                                            n_splits=FOLDS, repeats=REPEATS, seed=0,
                                            tag={'task': task, 'features': features})
            scores.append(s)
            params.append(p)
            if scheme == 'species':
                oof.append(o)
            print(f'{scheme:9s} {task:8s} {features:5s} done at {(time.perf_counter() - start) / 60:5.1f} min')

## Save

Everything goes to `results/`. The scores and the chosen settings come out identical on every run in the same environment; the timings don't, so they go to a separate file. The confusion counts pool the out-of-fold predictions over all three repeats of the species scheme.

In [ ]:
ids = ['scheme', 'task', 'features', 'model', 'repeat', 'fold']
fold_scores = pd.concat(scores, ignore_index=True)
fold_scores[ids + ['n_train', 'n_test'] + codon_utils.METRICS].to_csv(OUT / 'fold_scores.csv', index=False)
fold_scores[ids + ['seconds']].round(3).to_csv(OUT / 'timings.csv', index=False)
pd.concat(params, ignore_index=True)[ids + ['params']].to_csv(OUT / 'chosen_params.csv', index=False)

confusion = (pd.concat(oof, ignore_index=True)
             .groupby(['scheme', 'task', 'features', 'model', 'y_true', 'y_pred'])
             .size().rename('count').reset_index())
confusion.to_csv(OUT / 'confusion_counts.csv', index=False)
print(len(fold_scores), 'fold scores written to', OUT)

## First look

Before the careful comparisons in `05_results`, a first look: mean macro-F1 over the folds, with species held out. Uncertainty, paired tests, the leakage comparison and the figures are all in `05_results`.

In [ ]:
summary = codon_utils.summarize(fold_scores)
species = summary[summary['scheme'] == 'species']
species.pivot_table(index='model', columns=['task', 'features'], values='f1_macro_mean', sort=False).round(3)

## A check: is the "stop" group doing the work?

The exploration in `02_eda` turned up something that bears directly on the models. In animal mitochondria UGA encodes tryptophan, so in those rows the standard-code "stop" group mostly counts tryptophan. A single feature that differs that sharply between compartments could, in principle, be doing all the work of telling mitochondrial DNA from nuclear DNA, which would make the DNA-type results much less interesting than they look.

To find out, I refit k-NN and XGBoost (the two strongest DNA-type models) on the DNA-type task, species held out, on the same folds, with three feature sets: all 21 groups, the 20 groups without "stop", and "stop" alone. The first set is identical to the one in the main run, so its scores must reproduce the main run's exactly; the cell checks that they do, which also confirms that the folds are the same.

In [ ]:
import numpy as np

amino = FEATURES['amino']
ABLATION = {'amino': amino, 'amino_minus_stop': amino.drop(columns='stop'), 'stop_only': amino[['stop']]}
ablation_models = {name: models[name] for name in ['knn', 'xgb'] if name in models}
parts = []
for features, X in ABLATION.items():
    s, _, _ = codon_utils.nested_cv(X, TASKS['dna_type'], GROUPS['species'], ablation_models,
                                    scheme='species', n_splits=FOLDS, repeats=REPEATS, seed=0,
                                    tag={'task': 'dna_type', 'features': features})
    parts.append(s)
ablation = pd.concat(parts, ignore_index=True)

# the full 21-group runs must reproduce the main run exactly (same folds, same seeds)
main = fold_scores[(fold_scores['scheme'] == 'species') & (fold_scores['task'] == 'dna_type')
                   & (fold_scores['features'] == 'amino') & fold_scores['model'].isin(ablation_models)]
check = ablation[ablation['features'] == 'amino']
assert np.allclose(main.sort_values(ids)[codon_utils.METRICS], check.sort_values(ids)[codon_utils.METRICS], rtol=0, atol=1e-12)

ablation[ids + ['n_train', 'n_test'] + codon_utils.METRICS].to_csv(OUT / 'stop_ablation.csv', index=False)
ablation.pivot_table(index='model', columns='features', values='f1_macro', aggfunc='mean', sort=False).round(3)